# Python Notebook to Shift, Scale and Combine images

In [ ]:
# importing all the relevant libraries 
import numpy as np
import astroalign as aa
import ccdproc
import os
import astropy.io.fits as fits
from ccdproc import CCDData, combiner
from astropy import units as u
from astropy.time import Time
from photutils.aperture import CircularAperture
from photutils.aperture import aperture_photometry
import gc
gc.enable()

### Ensure the folders are setup correctly 
##### Create a Folder named "NGC_2547_sub"
##### Then inside that new folder, make three new folder named "b", "g", "r"

#### Input variables, select the star positions for each band

In [2]:
# positions of bright unsaturated stars
#                 star1x,star1y, star2x, star2y, star3x, star3y
positions_all = [[(603.44,1027), (807,863.08), (499.24, 855.88)], # b band positions
                 [(419, 811), (704, 871), (716, 1018)], # g band positions
                 [(418.6,807.4), (553, 1209.4), (704.2, 871)]  # r band positions
                 ]   

# selected appertures for each band
apertures_all = [15.0 , 10.0, 15.0]

# reference images for automatic alignment and scaling
aa_refidk = [0, 0, 0]

## Python Functions

### Band Selector Variable Assignments

In [3]:
def band_sel(fidx):
    # Band Variables
    bands = ["b", "g", "r"]

    band = bands[fidx]
    fn_name = '*_'+band + '.fit'
    refidx = aa_refidk[fidx]
    return fn_name, refidx, band

### Automatic Alignment

In [4]:
def autom_align(fidx): 

    # Generate data from band of interest
    fn_name, refidx, band = band_sel(fidx)
    print("Beginning Image Alignment for ", band, " images")

    # Collate images into scim and go into image path file
    images = ccdproc.ImageFileCollection("NGC_2547_sub/", glob_include=fn_name)
    os.chdir("NGC_2547_sub/")

    scim = []
    for fn in images.files_filtered():
        scim.append(CCDData.read(fn, unit = "adu")) 

        # Update scim with the correct time meta data
        times = scim[-1].header['DATE-OBS']                                      # Get the time
        t = Time(times, format='isot', scale='utc')                              # Read the time from UTC format
        scim[-1].header['MJD-OBS'] = (float(t.mjd), 'MJD')

    # Generate new file names
    newname = []          
    for fn in images.files_filtered():                                       
        newname.extend(["aa"+fn])                                            
    print("Aligned Image: ", newname, ) 

    # Navigate into band folder
    os.chdir(band + "/")
    for idx, thisimage in enumerate(scim):
        # Define images to be shifted
        # Plus zero is a quirky fix to an endian issue (old school)
        img_ref=scim[refidx].data+0       # Reference image
        img_example=scim[idx].data+0      # Image to be shifted image

        img_aligned, footprint = aa.register(img_example, img_ref, detection_sigma=5.0, min_area=9, fill_value=-99999.99)
        # print('Image alignment determined')

        temp=scim[idx]
        temp.data=img_aligned
        temp.meta = scim[idx].meta

        # Write the image to a file using the new name (aa- prefix)
        temp.write(newname[idx], overwrite=True)
    print("Image Alignment Completed")
        

### Scaling Function

In [5]:
def scaling(fidx):
    
    autom_align(fidx)

    # Generate data from band of interest
    fn_name, refidx, band = band_sel(fidx)
    positions = positions_all[fidx]
    print("Beginning Image Scaling for ", band, " images")

    images = ccdproc.ImageFileCollection(".",glob_include=fn_name)              # creating a collection 
    scim = []                                                                    # This is an empty list - scim
    for fn in images.files_filtered():
        scim.append(CCDData.read(fn, unit = "adu"))                              # reading images in the collection 
        times = scim[-1].header['DATE-OBS']                                      # Get the time
        t = Time(times, format='isot', scale='utc')                              # Read the time from UTC format
        scim[-1].header['MJD-OBS'] = (float(t.mjd), 'MJD')                       # Add MJD-OBS to header

    apertures = CircularAperture(positions, r=apertures_all[fidx])

    phot_tables=[]
    for idx, thisimage in enumerate(scim):                                      # Comment - summing the fluxes of the apertures of each image 
        phot_tables.extend([aperture_photometry(thisimage, apertures)])      

    print("Image Scaling Completed")
    return phot_tables

### Combine Function

In [6]:
def combine(fidx):
    phot_tables = scaling(fidx)

    # Generate data from band of interest
    fn_name, refidx, band = band_sel(fidx)
    print("Beginning Image Combining for ", band, " images")

    images = ccdproc.ImageFileCollection(".",glob_include = fn_name)          
    scim = [CCDData.read(fn) for fn in images.files_filtered()]                         

    # This code is missing comments. What is it doing?
    for idx, thisimage in enumerate(scim):                                              # iterating through each image        
        m = np.ma.median(phot_tables[refidx]['aperture_sum'] / phot_tables[idx]['aperture_sum'])    # scaling each image with the reference image
        print(m)
        scim[idx] = scim[idx].multiply(m * u.adu)                                       # normalising adu counts with reference image

    os.chdir("..")
    os.chdir("..")
    # Comment on the combine operations, including why minmax_clip_min = -500    
    sci_average = ccdproc.combine(scim, method = 'average',dtype = np.float32,          # mean combining images  
                                minmax_clip = True, minmax_clip_min = -500)           # rejecting any data with -500 adu counts in the average combining
    sci_average.write(band+"_NGC_2547_av.fits", overwrite=True)

    sci_median = ccdproc.combine(scim, method = 'median',dtype = np.float32,            # median combining images
                                minmax_clip = True, minmax_clip_min = -500)
    sci_median.write(band+"_NGC_2547_med.fits", overwrite=True)

    del(scim)
    collected = gc.collect()
    print('Check garbage collection', collected)

    print(band, " Images Combining Finished")

## Executing Code
This executions completes shifting, scaling and combining for all bands. 

In [7]:
for fidx in list(range(3)):
    combine(fidx)

Beginning Image Alignment for  b  images
Aligned Image:  ['aaLight_NGC_2547_10.0s_IRCUT_20260331-200329_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200351_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200403_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200415_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200425_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200437_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200500_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200512_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200524_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200534_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200546_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200616_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200627_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200639_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200651_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200702_b.fit', 'aaLight_NGC_2547_10.0s_IRCUT_20260331-200748_b.fit', 'aaLight_NGC_2547_10.0s_

INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.


INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader inste

INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.


INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader inste

...

INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader inste

INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.
INFO:astropy:using the unit adu passed to the FITS reader instead of the unit adu in the FITS file.


INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
INFO: using the unit adu passed to the FITS reader inste

INFO:astropy:splitting each image into 2 chunks to limit memory usage to 16000000000.0 bytes.


INFO: splitting each image into 2 chunks to limit memory usage to 16000000000.0 bytes. [ccdproc.combiner]
Check garbage collection 1696
r  Images Combining Finished
